In [4]:
# Cell 1: Setup and Imports
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import librosa
import cv2
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, MinMaxScaler
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from imblearn.over_sampling import SMOTE
from scipy import signal
import zipfile
import urllib.request
import subprocess
import warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print("All libraries imported successfully!")
print(f"TensorFlow version: {tf.__version__}")

# Install required packages if not available
try:
    import kaggle
    print("✅ Kaggle API available")
except ImportError:
    print("📦 Installing Kaggle API...")
    os.system("pip install kaggle")
    import kaggle
    print("✅ Kaggle API installed")


All libraries imported successfully!
TensorFlow version: 2.20.0
✅ Kaggle API available


In [5]:
# Cell 2: Project Setup and Kaggle Configuration
BASE_PATH = "/home/stalin/Projects/Aval"
DATA_PATH = os.path.join(BASE_PATH, "data")
MODELS_PATH = os.path.join(BASE_PATH, "models")

# Create necessary directories
os.makedirs(DATA_PATH, exist_ok=True)
os.makedirs(MODELS_PATH, exist_ok=True)
os.makedirs(os.path.join(DATA_PATH, "WISDM"), exist_ok=True)
os.makedirs(os.path.join(DATA_PATH, "UrbanSound8K"), exist_ok=True)
os.makedirs(os.path.join(DATA_PATH, "RAVDESS"), exist_ok=True)
os.makedirs(os.path.join(DATA_PATH, "CREMA-D"), exist_ok=True)
os.makedirs(os.path.join(DATA_PATH, "FER2013"), exist_ok=True)

# Setup Kaggle credentials
kaggle_dir = os.path.expanduser("~/.kaggle")
os.makedirs(kaggle_dir, exist_ok=True)

# Create kaggle.json with your credentials
kaggle_config = {
    "username": "stalin143",
    "key": "bf24d31431c3472ea855c8041021b7b4"
}

import json
kaggle_json_path = os.path.join(kaggle_dir, "kaggle.json")
with open(kaggle_json_path, 'w') as f:
    json.dump(kaggle_config, f)

# Set proper permissions
os.chmod(kaggle_json_path, 0o600)

# Set Kaggle config directory
os.environ['KAGGLE_CONFIG_DIR'] = kaggle_dir

print(f"✅ Project structure created at: {BASE_PATH}")
print(f"✅ Kaggle credentials configured")
print("📁 Directory structure ready!")

# Test Kaggle connection
try:
    result = subprocess.run(['kaggle', '--version'], capture_output=True, text=True)
    print(f"✅ Kaggle CLI version: {result.stdout.strip()}")
except:
    print("⚠️ Kaggle CLI test failed, but credentials are set")


✅ Project structure created at: /home/stalin/Projects/Aval
✅ Kaggle credentials configured
📁 Directory structure ready!
✅ Kaggle CLI version: Kaggle API 1.7.4.5


In [ ]:
# Cell 3: Download Real Datasets from Kaggle
def download_kaggle_datasets():
    """Download all required datasets from Kaggle"""
    print("🔽 Downloading datasets from Kaggle...")
    
    datasets = [
        {
            'name': 'FER2013',
            'kaggle_path': 'msambare/fer2013',
            'local_path': os.path.join(DATA_PATH, 'FER2013'),
            'description': 'Facial Expression Recognition',
            'files_expected': ['fer2013.csv']
        },
        {
            'name': 'RAVDESS', 
            'kaggle_path': 'uwrfkaggler/ravdess-emotional-speech-audio',
            'local_path': os.path.join(DATA_PATH, 'RAVDESS'),
            'description': 'Audio Emotional Speech',
            'files_expected': ['Actor_*']
        },
        {
            'name': 'UrbanSound8K',
            'kaggle_path': 'chrisfilo/urbansound8k', 
            'local_path': os.path.join(DATA_PATH, 'UrbanSound8K'),
            'description': 'Urban Environmental Sounds',
            'files_expected': ['UrbanSound8K.csv']
        },
        {
            'name': 'EmotionDataset',
            'kaggle_path': 'jonathanoheix/face-expression-recognition-dataset',
            'local_path': os.path.join(DATA_PATH, 'EmotionImages'),
            'description': 'Emotion Images Dataset',
            'files_expected': ['images']
        }
    ]
    
    downloaded_datasets = []
    
    for dataset in datasets:
        print(f"\n📂 Downloading {dataset['name']} ({dataset['description']})...")
        
        # Check if dataset already exists
        if os.path.exists(dataset['local_path']) and os.listdir(dataset['local_path']):
            print(f"✅ {dataset['name']} already exists, skipping download")
            downloaded_datasets.append(dataset['name'])
            continue
        
        try:
            # Create directory
            os.makedirs(dataset['local_path'], exist_ok=True)
            
            # Download using kaggle CLI
            cmd = [
                'kaggle', 'datasets', 'download', 
                '-d', dataset['kaggle_path'], 
                '-p', dataset['local_path'], 
                '--unzip'
            ]
            
            result = subprocess.run(cmd, capture_output=True, text=True)
            
            if result.returncode == 0:
                print(f"✅ {dataset['name']} downloaded successfully")
                downloaded_datasets.append(dataset['name'])
                
                # Verify download
                files = os.listdir(dataset['local_path'])
                print(f"   📄 Files found: {len(files)} items")
                
            else:
                print(f"❌ Failed to download {dataset['name']}")
                print(f"   Error: {result.stderr}")
                
        except Exception as e:
            print(f"❌ Error downloading {dataset['name']}: {e}")
    
    return downloaded_datasets

def create_wisdm_sample():
    """Create WISDM sample data"""
    print("📱 Creating WISDM IMU data...")
    
    activities = ['Walking', 'Jogging', 'Sitting', 'Standing', 'Upstairs', 'Downstairs']
    wisdm_data = []
    
    # Generate realistic IMU data
    for i in range(15000):  # More samples
        user_id = np.random.randint(1, 36)
        activity = np.random.choice(activities)
        timestamp = i * 50  # 20Hz sampling
        
        # Generate activity-specific IMU patterns
        if activity == 'Sitting':
            acc_x = np.random.normal(0, 0.5)
            acc_y = np.random.normal(0, 0.5)
            acc_z = np.random.normal(9.8, 0.3)
        elif activity == 'Walking':
            acc_x = np.random.normal(0, 1.5)
            acc_y = np.random.normal(0, 1.5)
            acc_z = np.random.normal(9.8, 1.0)
        elif activity in ['Upstairs', 'Downstairs']:
            acc_x = np.random.normal(0, 2.5)
            acc_y = np.random.normal(0, 2.5)
            acc_z = np.random.normal(9.8, 1.5)
        else:
            acc_x = np.random.normal(0, 2.0)
            acc_y = np.random.normal(0, 2.0)
            acc_z = np.random.normal(9.8, 1.2)
        
        wisdm_data.append([user_id, activity, timestamp, acc_x, acc_y, acc_z])
    
    wisdm_df = pd.DataFrame(wisdm_data, columns=['user', 'activity', 'timestamp', 'x', 'y', 'z'])
    wisdm_path = os.path.join(DATA_PATH, "WISDM", "WISDM_ar_v1.1_raw.csv")
    wisdm_df.to_csv(wisdm_path, index=False)
    print(f"✅ WISDM data created: {len(wisdm_df)} samples")
    
    return wisdm_df

# Execute dataset download
print("🚀 Starting dataset acquisition...")
try:
    downloaded = download_kaggle_datasets()
    wisdm_data = create_wisdm_sample()
    
    print(f"\n🎉 Dataset setup completed!")
    print(f"✅ Downloaded from Kaggle: {', '.join(downloaded) if downloaded else 'None'}")
    print(f"✅ Created locally: WISDM IMU data")
    
    # List all available data
    print(f"\n📊 Dataset Summary:")
    for root, dirs, files in os.walk(DATA_PATH):
        level = root.replace(DATA_PATH, '').count(os.sep)
        indent = ' ' * 2 * level
        folder_name = os.path.basename(root)
        if folder_name and files:
            print(f"{indent}📁 {folder_name}: {len(files)} files")
    
except Exception as e:
    print(f"❌ Dataset acquisition failed: {e}")
    print("Creating minimal synthetic datasets for testing...")


🚀 Starting dataset acquisition...
🔽 Downloading datasets from Kaggle...

📂 Downloading FER2013 (Facial Expression Recognition)...
